In [ ]:
!pip install -q cobra torch_geometric pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.1 MB/s eta 0:00:00


In [ ]:
import cobra
import torch_geometric
import pandas as pd
import numpy as np

print("COBRApy version:", cobra.__version__)
print("PyTorch Geometric version:", torch_geometric.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("\nAll libraries imported successfully!")

COBRApy version: 0.32.1
PyTorch Geometric version: 2.8.0.post1
Pandas version: 2.2.3
NumPy version: 2.1.3

All libraries imported successfully!


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
import cobra

print("=== STEP [1]: Multi-Omics Data Setup ===")
# Create 5 synthetic rice gene IDs matching standard RAP-DB format
gene_ids = ['Os01g0100100', 'Os01g0100200', 'Os01g0100300', 'Os01g0100400', 'Os01g0100500']

# Simulated RNA-seq expression across Control, Drought, and Salt
exp_data = pd.DataFrame({
    'Gene_ID': gene_ids,
    'Control': [10.2, 45.1, 2.3, 88.0, 12.1],
    'Drought': [85.4, 40.0, 1.2, 120.5, 11.8],
    'Salt':    [60.1, 42.8, 0.5, 95.2, 14.0]
})
print(exp_data)


print("\n=== STEP [2]: PyTorch Geometric Graph Setup ===")
# 1. Map gene string names to matrix index numbers (0 to 4)
gene2idx = {gene: idx for idx, gene in enumerate(gene_ids)}

# 2. Convert expression values into Node Features (x)
x_features = torch.tensor(exp_data[['Control', 'Drought', 'Salt']].values, dtype=torch.float)

# 3. Define Gene Interaction Edges (edge_index)
# Example interaction pairs: (Os01g0100100 <-> Os01g0100200), (Os01g0100200 <-> Os01g0100300), etc.
src = [gene2idx['Os01g0100100'], gene2idx['Os01g0100200'], gene2idx['Os01g0100300'], gene2idx['Os01g0100400']]
dst = [gene2idx['Os01g0100200'], gene2idx['Os01g0100300'], gene2idx['Os01g0100400'], gene2idx['Os01g0100500']]
edge_index = torch.tensor([src, dst], dtype=torch.long)

# Build PyG Graph Data Object
rice_graph = Data(x=x_features, edge_index=edge_index)
print("PyG Graph Object Created:", rice_graph)


print("\n=== STEPS [3] & [4]: COBRApy Metabolic Knockout & >5% Flux Rule ===")
# Load standard benchmark metabolic model to test the flux logic
model = cobra.io.load_model("textbook")

# Calculate baseline wild-type growth/biomass rate
wt_rate = model.optimize().objective_value
print(f"Wild-Type Biomass Rate: {wt_rate:.4f}")

# Simulate knockout for target gene 'b0008' (example gene in benchmark model)
with model:
    model.genes.b0008.knockout()
    ko_rate = model.optimize().objective_value

# Calculate percentage flux change
flux_change_pct = abs((wt_rate - ko_rate) / wt_rate) * 100
print(f"Knockout Flux Rate: {ko_rate:.4f} | Change: {flux_change_pct:.2f}%")

# Decision Checkpoint
if flux_change_pct > 5.0:
    print("Decision: Flux change > 5% -> PASS (Proceed to Step [5] Parameter Mapping)")
else:
    print("Decision: Flux change <= 5% -> DROP candidate")

COBRApy version: 0.32.1
Wild-Type Biomass Rate: 0.8739
Knockout (b0008) Biomass Rate: 0.8739
Relative growth: 100.00% of wild type
       ids    growth   status
0  {b0008}  0.873922  optimal
Model restored, growth: 0.8739


In [ ]:
from cobra.flux_analysis import single_gene_deletion
import pandas as pd

# 1. Run single gene deletion analysis across ALL genes in the model
print("Screening all genes in the model for flux perturbations...")
deletion_results = single_gene_deletion(model)

# 2. Calculate percentage change in biomass/growth rate relative to Wild-Type
wt_growth = wt_solution.objective_value
deletion_results['flux_change_pct'] = (
    abs(wt_growth - deletion_results['growth']) / wt_growth
) * 100

# 3. Apply your Decision Rule: Flux Change > 5%
# (Excluding total lethality/zero growth if you only want non-lethal perturbations)
candidate_targets = deletion_results[
    (deletion_results['flux_change_pct'] > 5.0) &
    (deletion_results['growth'] > 0.001)
].copy()

print(f"\n--- SCREENING RESULTS ---")
print(f"Total Genes Screened: {len(deletion_results)}")
print(f"Candidate Targets Passing >5% Flux Threshold: {len(candidate_targets)}")

# Display top candidate genes that passed the filter
candidate_targets.sort_values(by='flux_change_pct', ascending=False).head(10)

Screening all genes in the model for flux perturbations...

--- SCREENING RESULTS ---
Total Genes Screened: 137
Candidate Targets Passing >5% Flux Threshold: 31


,ids,growth,status,flux_change_pct
8,{s0001},0.211141,optimal,75.839861
75,{b2285},0.211663,optimal,75.780096
49,{b2284},0.211663,optimal,75.780096
91,{b2283},0.211663,optimal,75.780096
60,{b2282},0.211663,optimal,75.780096
88,{b2277},0.211663,optimal,75.780096
97,{b2278},0.211663,optimal,75.780096
109,{b2287},0.211663,optimal,75.780096
115,{b2279},0.211663,optimal,75.780096
103,{b2286},0.211663,optimal,75.780096


In [ ]:
# Baseline Rice Parameters for APSIM
BASELINE_RUE = 1.25  # g/MJ (Radiation Use Efficiency)
BASELINE_TE = 6.0    # g/kg (Transpiration Efficiency)

def map_flux_to_crop_params(flux_change_pct, is_positive_knockout=True):
    """
    Translates metabolic flux delta % into scaled crop parameters for APSIM.
    """
    # Scale factor (e.g., 10% flux increase -> 2% boost in RUE)
    scaling_factor = 0.2

    delta_ratio = (flux_change_pct / 100.0) * scaling_factor

    new_rue = BASELINE_RUE * (1 + delta_ratio)
    new_te = BASELINE_TE * (1 + delta_ratio)

    return round(new_rue, 4), round(new_te, 4)

# Apply mapping to your candidate targets dataframe
candidate_targets['APSIM_RUE'] = candidate_targets['flux_change_pct'].apply(
    lambda pct: map_flux_to_crop_params(pct)[0]
)
candidate_targets['APSIM_TE'] = candidate_targets['flux_change_pct'].apply(
    lambda pct: map_flux_to_crop_params(pct)[1]
)

print("Mapped Crop Parameters for Candidate Targets:")
candidate_targets[['growth', 'flux_change_pct', 'APSIM_RUE', 'APSIM_TE']].head()

Mapped Crop Parameters for Candidate Targets:


,growth,flux_change_pct,APSIM_RUE,APSIM_TE
1,0.825819,5.504180,1.2638,6.0661
6,0.704037,19.439338,1.2986,6.2333
8,0.211141,75.839861,1.4396,6.9101
9,0.211663,75.780096,1.4395,6.9094
18,0.211663,75.780096,1.4395,6.9094


In [ ]:
import numpy as np
import pandas as pd

# 1. Re-define baseline crop yield simulation parameters
BASELINE_RUE = 1.25  # Radiation Use Efficiency
BASELINE_TE = 6.0    # Transpiration Efficiency

def simulate_crop_yield(rue, te, days=120):
    np.random.seed(42)
    daily_solar_radiation = np.random.uniform(15, 25, days)
    daily_water_supply = np.random.uniform(1.0, 5.0, days)
    total_biomass = 0.0
    for day in range(days):
        biomass_light = daily_solar_radiation[day] * rue
        biomass_water = daily_water_supply[day] * te
        total_biomass += min(biomass_light, biomass_water)
    return round(total_biomass * 0.45, 2)

baseline_yield = simulate_crop_yield(BASELINE_RUE, BASELINE_TE)

# 2. Ensure APSIM parameters exist in candidate_targets
if 'APSIM_RUE' not in candidate_targets.columns:
    candidate_targets['APSIM_RUE'] = candidate_targets['flux_change_pct'].apply(
        lambda pct: BASELINE_RUE * (1 + (pct / 100.0) * 0.2)
    )

# 3. Calculate Simulated Yield & Yield Gain Pct directly
candidate_targets['Simulated_Yield_g_m2'] = candidate_targets['APSIM_RUE'].apply(
    lambda rue: simulate_crop_yield(rue, BASELINE_TE)
)
candidate_targets['Yield_Gain_Pct'] = (
    (candidate_targets['Simulated_Yield_g_m2'] - baseline_yield) / baseline_yield
) * 100

# 4. Step [7] Multi-Criteria Ranking
candidate_targets['Rank_Score'] = (
    (candidate_targets['Yield_Gain_Pct'] * 0.7) +
    (candidate_targets['flux_change_pct'] * 0.3)
)

# Extract Top 5 Target Edits
top_5_targets = candidate_targets.sort_values(
    by='Rank_Score', ascending=False
).head(5).copy()

print("=== STEP [7]: TOP 5 RANKED TARGET EDITS ===")
display(top_5_targets[['growth', 'flux_change_pct', 'Yield_Gain_Pct', 'Rank_Score']])

=== STEP [7]: TOP 5 RANKED TARGET EDITS ===


,growth,flux_change_pct,Yield_Gain_Pct,Rank_Score
8,0.211141,75.839861,3.556448,25.241472
75,0.211663,75.780096,3.555332,25.222762
49,0.211663,75.780096,3.555332,25.222762
91,0.211663,75.780096,3.555332,25.222762
9,0.211663,75.780096,3.555332,25.222762


In [ ]:
import re

def design_cas9_grna(gene_id, mock_dna_sequence):
    """
    Scans a target DNA sequence for SpCas9 PAM sites (NGG)
    and extracts 20-nt protospacer guide RNA targets.
    """
    grnas = []
    # Find all NGG motifs in the sequence
    # Cas9 looks for a 20-nt spacer directly upstream of NGG (5' -> 3')
    pattern = r'(?=([ATCG]{20}[ATCG]GG))'

    matches = re.finditer(pattern, mock_dna_sequence)
    for i, match in enumerate(matches):
        full_match = match.group(1)
        spacer = full_match[:20]  # 20-nt gRNA
        pam = full_match[20:]     # NGG PAM site

        # Calculate GC content (ideal gRNA GC content is 40% - 60%)
        gc_content = ((spacer.count('G') + spacer.count('C')) / 20.0) * 100

        grnas.append({
            'Gene_ID': gene_id,
            'gRNA_ID': f"{gene_id}_gRNA_{i+1}",
            'gRNA_Sequence_5_to_3': spacer,
            'PAM': pam,
            'GC_Content_Pct': gc_content
        })
    return grnas

# Mock genomic DNA sequence for demonstration
sample_gene_sequence = "ATGCGATCGATCGATCGATCGATCGATCGACTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGNGGATCGATCGATCGATCGATCG"

# Generate CRISPR Guides for the #1 Top Ranked Target
top_gene_id = str(top_5_targets.index[0])
designed_guides = design_cas9_grna(top_gene_id, sample_gene_sequence)

import pandas as pd
df_grnas = pd.DataFrame(designed_guides)

print(f"=== STEP [8]: DESIGNED CRISPR gRNAs FOR TOP TARGET {top_gene_id} ===")
df_grnas.head()

=== STEP [8]: DESIGNED CRISPR gRNAs FOR TOP TARGET 8 ===


""
